# De la predicción a la evidencia: comparar y auditar analizadores de sentimiento

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta central

¿Qué tan defendibles son las clasificaciones de nuestros tres métodos cuando los aplicamos a este corpus?

## Objetivo

Aplicar los tres métodos de los cuadernos anteriores a un mismo conjunto de textos auténticos, compararlos con una referencia humana construida por el grupo y justificar con evidencia textual qué se puede y qué no se puede afirmar.

## Al terminar este cuaderno vas a poder:

1. Anotar textos con un protocolo explícito y documentar los casos que no se pueden clasificar de forma defendible.
2. Formular una pregunta y una hipótesis contrastable con los datos disponibles.
3. Distinguir tres cosas: la predicción de un método, la evaluación contra una referencia y la interpretación de un caso.
4. Auditar desacuerdos y errores con evidencia textual.
5. Redactar una conclusión que diga qué permiten y qué no permiten afirmar estos datos.

## Producto

Una tabla comparativa de los tres métodos sobre el mismo corpus, el análisis escrito de al menos tres casos y un ensayo computacional breve.

## Relación con los tres cuadernos anteriores

| Cuaderno | Método | Cómo se usa acá |
|---|---|---|
| 01 | **Método A**: léxico de 27 palabras y una regla de negación | Mismo léxico, misma regla, mismo criterio para pasar de puntuación a etiqueta. |
| 02 | **Método B**: TF-IDF y regresión logística | Entrenado **solo** con las 24 oraciones construidas del cuaderno 02. El corpus nuevo no interviene en el ajuste. |
| 03 | **Método C**: modelo preentrenado de `pysentimiento` | Mismo modelo, aplicado sin reentrenar. |

Este cuaderno es **autónomo**: copia las funciones necesarias (con su procedencia) y no depende de variables ni de ejecuciones de los otros cuadernos.

Los tres métodos no están ordenados de peor a mejor. Cada uno hace visibles unas decisiones y esconde otras, y el objetivo es averiguar qué evidencia sostiene cada clasificación.

## Alcance y límites

- El corpus es una **selección docente de entre 30 y 60 fragmentos** de un mismo género. Esa escala permite leer y revisar cada caso. **No** garantiza representatividad ni validez estadística.
- No hacemos pruebas de significación ni afirmaciones sobre poblaciones. Los resultados describen estos fragmentos, estos métodos y esta anotación.
- No se introduce ninguna técnica nueva.

## Corpus construido y corpus auténtico

| | Corpus construido (cuadernos 01 a 03) | Corpus auténtico (este cuaderno) |
|---|---|---|
| Quién lo escribió | El docente, para la actividad | Medios periodísticos, para sus lectores |
| Para qué se escribió | Mostrar un mecanismo | Informar, opinar, titular |
| Qué permite | Ver cómo funciona un método | Ver cómo se comporta un método con textos que no se hicieron para él |
| Qué no permite | Estimar rendimiento | Generalizar más allá de la selección |

Un corpus auténtico no es automáticamente más válido: tiene procedencia, recorte y anotación que hay que examinar.

## ◈ Microglosario

| Término | Qué significa en este cuaderno |
|---|---|
| **Unidad de análisis** | Lo que cuenta como una observación: una fila de la tabla. Hay que declararla. |
| **Valoración dirigida** | Valoración que se asigna a un objeto concreto (por ejemplo, una persona o una institución), no al texto entero. |
| **Protocolo de anotación** | Reglas escritas que dicen cuándo corresponde `POS`, `NEG` o `NEU`. |
| **Referencia humana** | Etiquetas acordadas por el grupo con ese protocolo. Es una vara de comparación, no una verdad. |
| **Acuerdo entre métodos** | Dos o más métodos dan la misma etiqueta. No implica que sea correcta. |
| **Macro-F1** | Promedio simple del F1 de cada clase, sin ponderar por cantidad de ejemplos. Solo se calcula con las clases que tienen ejemplos de referencia. |

## Configuración

Todas las rutas y umbrales de este cuaderno están en la celda siguiente. Si cambiás de corpus, es lo único que tenés que tocar.

**Descarga del modelo.** El método C usa un modelo que se descarga de Internet la primera vez (ver `ANTES_DE_LA_CLASE.md`). Ningún texto del corpus se envía a servicios externos: el modelo se ejecuta en tu computadora.

In [1]:
import sys
import importlib.metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# utils.py (en esta carpeta): ayudas que dibujan los gráficos. No calculan nada.
from utils import (
    ETIQUETAS, COLOR_GENERAL,
    dibujar_barras, dibujar_matriz_de_confusión, dibujar_probabilidades,
)

# Corpus de trabajo (selección docente). Si falta, el cuaderno se detiene con un mensaje.
RUTA_CORPUS = Path("datos/titulares_2019.csv")

# Corpus construido del cuaderno 02. Se usa SOLO para entrenar el método B.
RUTA_ENTRENAMIENTO = Path("datos/corpus_sentimiento.csv")

# Anotaciones acordadas por el grupo. Se crea después de la actividad en parejas.
RUTA_ANOTACIONES = Path("anotaciones/anotaciones_acordadas.csv")

# Etiqueta original del corpus. Solo se lee en la sección 10, después de anotar.
RUTA_REFERENCIA_ORIGINAL = Path("datos/_referencia_corpus_original.csv")

# Carpeta donde este cuaderno guarda lo que genera.
CARPETA_SALIDAS = Path("salidas")

# Umbrales de trabajo. Son decisiones declaradas, no criterios estadísticos.
MINIMO_EVALUADOS = 10             # con menos registros anotados no calculamos métricas
UMBRAL_PROBABILIDAD_ALTA = 0.80   # para la tabla de desacuerdos de pysentimiento

pd.set_option("display.max_colwidth", 140)

print("Python:", sys.version.split()[0])
for paquete in ["pandas", "scikit-learn", "matplotlib"]:
    print(f"{paquete}: {importlib.metadata.version(paquete)}")

Python: 3.12.13
pandas: 3.0.6
scikit-learn: 1.9.1
matplotlib: 3.11.2


**Leé el resultado.** Tienen que aparecer las tres versiones. Las versiones de `torch`, `transformers` y `pysentimiento` se registran más adelante, cuando se carga el modelo.

## 2. Procedencia y delimitación

**Pregunta.** ¿Qué textos tenemos, de dónde vienen y qué problemas tienen antes de analizarlos?

**Contrato de entrada.** El CSV del corpus necesita estas columnas:

| Columna | Obligatoria | Qué contiene |
|---|---|---|
| `id` | sí | Identificador único de cada fragmento. Con él se unen las anotaciones: **no** se unen por posición. |
| `texto` | sí | El fragmento, tal como se va a leer y a analizar. |
| `fuente` | sí | De dónde salió, en una frase. |
| `id_documento` | sí | Documento al que pertenece el fragmento. Varios fragmentos pueden compartirlo. |
| `fecha`, `contexto`, `objeto_valorado` | no | Si existen, se conservan. `contexto` y `objeto_valorado` se muestran a quien anota. |

Hay una plantilla vacía, solo con encabezados, en `datos/plantilla_corpus.csv`.

**Decisión.** Si el archivo falta, el cuaderno se detiene. No continúa con datos de ejemplo ni sintéticos.

In [2]:
if not RUTA_CORPUS.exists():
    raise FileNotFoundError(
        f"Falta el corpus de trabajo: {RUTA_CORPUS}\n"
        "Preparalo como un CSV con las columnas id, texto, fuente, id_documento "
        "(opcionales: fecha, contexto, objeto_valorado).\n"
        "Hay una plantilla con los encabezados en datos/plantilla_corpus.csv y las "
        "instrucciones están en el README de esta carpeta."
    )

if not RUTA_ENTRENAMIENTO.exists():
    raise FileNotFoundError(
        f"Falta el corpus de entrenamiento del método B: {RUTA_ENTRENAMIENTO}\n"
        "Es el archivo datos/corpus_sentimiento.csv del cuaderno 02."
    )

COLUMNAS_REQUERIDAS = ["id", "texto", "fuente", "id_documento"]

# dtype=str lee todo como texto, para que un identificador como 007 no pierda sus ceros.
# keep_default_na=False evita que un campo vacío se convierta en un valor faltante.
corpus_original = pd.read_csv(RUTA_CORPUS, dtype=str, keep_default_na=False, encoding="utf-8-sig")

columnas_faltantes = []
for columna in COLUMNAS_REQUERIDAS:
    if columna not in corpus_original.columns:
        columnas_faltantes.append(columna)
if columnas_faltantes:
    raise ValueError(
        f"Al corpus le faltan columnas requeridas: {columnas_faltantes}. "
        f"Columnas encontradas: {list(corpus_original.columns)}."
    )

print(f"Corpus leído de {RUTA_CORPUS}: {len(corpus_original)} filas")
print("Columnas:", list(corpus_original.columns))
display(corpus_original.head())

Corpus leído de datos/titulares_2019.csv: 48 filas
Columnas: ['id', 'texto', 'fuente', 'id_documento', 'objeto_valorado']


,id,texto,fuente,id_documento,objeto_valorado
0,T01,Alfredo Cornejo marcó distancia de Mauricio Macri y apuntó contra Alberto Fernández: “Vuelven peores que antes”,"pysentimiento/spanish-targeted-sentiment-headlines (Hugging Face), partición test; titulares de medios argentinos, elecciones presidenci...",1754,Mauricio Macri
1,T02,Mauricio Macri y otros dirigentes de Cambiemos celebraron la victoria de Rodolfo Suárez en Mendoza,"pysentimiento/spanish-targeted-sentiment-headlines (Hugging Face), partición test; titulares de medios argentinos, elecciones presidenci...",1762,Cambiemos
2,T03,¿Está terminado Mauricio Macri?,"pysentimiento/spanish-targeted-sentiment-headlines (Hugging Face), partición test; titulares de medios argentinos, elecciones presidenci...",2046,Mauricio Macri
3,T04,José López se negó a declarar en el juicio por la obra pública en el que está acusado junto a Cristina Kirchner,"pysentimiento/spanish-targeted-sentiment-headlines (Hugging Face), partición test; titulares de medios argentinos, elecciones presidenci...",2719,Cristina Kirchner
4,T05,Macri lanza su primera propuesta de campaña: bajar impuestos al trabajo,"pysentimiento/spanish-targeted-sentiment-headlines (Hugging Face), partición test; titulares de medios argentinos, elecciones presidenci...",2829,Macri


**Leé el resultado.** El archivo original **no se modifica** en ningún momento: a partir de acá trabajamos con una copia.

### Validaciones

Antes de analizar, revisamos cinco cosas. Ninguna fila se elimina en silencio: lo que aparezca se muestra y se explica qué se decide.

In [4]:
corpus = corpus_original.copy()

# Un texto "vacío" es uno que solo tiene espacios. strip() quita los espacios de los extremos.
ids_vacíos = corpus["id"].str.strip() == ""
ids_repetidos = corpus["id"].duplicated(keep=False)
textos_vacíos = corpus["texto"].str.strip() == ""

# Un duplicado exacto repite el texto Y el objeto valorado (el mismo titular con dos objetos distintos no lo es).
columnas_del_duplicado = ["texto"]
if "objeto_valorado" in corpus.columns:
    columnas_del_duplicado.append("objeto_valorado")
duplicados_exactos = corpus.duplicated(subset=columnas_del_duplicado, keep=False)

controles = pd.DataFrame([
    {"control": "Identificadores vacíos", "filas": int(ids_vacíos.sum()), "qué se decide": "Si hay alguno, el cuaderno se detiene: sin id no se pueden unir las anotaciones."},
    {"control": "Identificadores repetidos", "filas": int(ids_repetidos.sum()), "qué se decide": "Si hay alguno, el cuaderno se detiene por la misma razón."},
    {"control": "Textos vacíos", "filas": int(textos_vacíos.sum()), "qué se decide": "Se conservan en la tabla, pero no se clasifican ni se evalúan. Quedan marcados."},
    {"control": "Duplicados exactos", "filas": int(duplicados_exactos.sum()), "qué se decide": "Se conservan y se muestran. Decidí vos si corresponde revisarlos."},
])
display(controles)

if ids_vacíos.any() or ids_repetidos.any():
    display(corpus.loc[ids_vacíos | ids_repetidos])
    raise ValueError("Hay identificadores vacíos o repetidos: corregilos en el CSV antes de seguir.")

if textos_vacíos.any():
    display(corpus.loc[textos_vacíos])
if duplicados_exactos.any():
    display(corpus.loc[duplicados_exactos])

,control,filas,qué se decide
0,Identificadores vacíos,0,"Si hay alguno, el cuaderno se detiene: sin id no se pueden unir las anotaciones."
1,Identificadores repetidos,0,"Si hay alguno, el cuaderno se detiene por la misma razón."
2,Textos vacíos,0,"Se conservan en la tabla, pero no se clasifican ni se evalúan. Quedan marcados."
3,Duplicados exactos,0,Se conservan y se muestran. Decidí vos si corresponde revisarlos.


**Registro de exclusiones y conteos.** Preparamos la columna que explica por qué una fila no se evalúa y contamos fragmentos y documentos.

In [5]:
# Columna que dice por qué una fila no entra en la clasificación o en la evaluación.
corpus["motivo_exclusion"] = ""
corpus.loc[textos_vacíos, "motivo_exclusion"] = "texto vacío"

cantidad_de_fragmentos = len(corpus)
cantidad_de_documentos = corpus["id_documento"].nunique()

# groupby agrupa las filas por documento; count cuenta cuántos fragmentos tiene cada uno.
agrupado_por_documento = corpus.groupby("id_documento")
fragmentos_por_documento = agrupado_por_documento["id"].count()
documentos_con_varios = fragmentos_por_documento[fragmentos_por_documento > 1]

print(f"Fragmentos (filas): {cantidad_de_fragmentos}")
print(f"Documentos distintos (id_documento): {cantidad_de_documentos}")
print(f"Documentos con más de un fragmento: {len(documentos_con_varios)}")

Fragmentos (filas): 48
Documentos distintos (id_documento): 48
Documentos con más de un fragmento: 0


**Leé el resultado.** Anotá cuántos fragmentos y cuántos documentos hay.

- Si hay documentos con varios fragmentos, esos fragmentos **no son independientes**: comparten autor, tema y contexto, y contarlos como observaciones separadas puede dar una impresión de mayor evidencia de la que hay.
- Si cada fragmento viene de un documento distinto, esa dependencia no aparece en esta selección, pero la pregunta sigue valiendo para otros recortes.

### Procedencia declarada por la cátedra para el corpus entregado

Si usás otro corpus, reemplazá este bloque.

| Dato | Contenido |
|---|---|
| Fuente | Dataset `pysentimiento/spanish-targeted-sentiment-headlines` en Hugging Face ([enlace](https://huggingface.co/datasets/pysentimiento/spanish-targeted-sentiment-headlines)). |
| Descripción de origen | Titulares de medios argentinos sobre las elecciones presidenciales de 2019, descriptos en el artículo [arXiv 2208.13947](https://arxiv.org/abs/2208.13947). |
| Género | Titulares periodísticos de tema político. |
| Recorte | Selección con semilla declarada, hecha con `datos/preparar_titulares.py`: ver el campo `fuente` de cada fila. |
| Unidad de análisis | Un **titular y un objeto valorado** (`objeto_valorado`). Un mismo titular puede aparecer con objetos distintos. |
| Qué no se sabe | El repositorio no incluye una ficha de datos. Quién anotó las etiquetas originales, con qué criterio y con qué acuerdo entre anotadores no está detallado en lo que consultamos. |
| Licencia | Verificar antes de redistribuir los textos: el artículo indica CC BY 4.0, pero el repositorio no tiene ficha propia. |
| Privacidad | Titulares ya publicados que mencionan figuras públicas. No contienen datos personales de personas privadas, pero el tema es políticamente sensible. |

### Ficha editable del grupo

| Pregunta | Nuestra respuesta |
|---|---|
| ¿De dónde provienen los textos? | |
| ¿Por qué fueron incluidos? | |
| ¿Qué quedó fuera? | |
| ¿Qué constituye una observación? | |
| ¿Qué población o conjunto documental **no** representan? | |

**Aviso sobre el tema.** Este cuaderno analiza el comportamiento de métodos de clasificación. No es un análisis de los candidatos, de los medios ni del electorado.

## 3. Lectura y anotación antes de las predicciones

**Regla del cuaderno.** La primera tabla que ves no muestra ninguna salida automática. Primero leés y anotás; después se comparan los métodos.

### Protocolo inicial (editable)

Podés ajustarlo con tu grupo antes de anotar. Si lo cambiás, escribí el cambio y la razón.

- `POS`: valoración favorable explícita.
- `NEG`: valoración desfavorable explícita.
- `NEU`: información sin valoración explícita según este protocolo.

**Casos que no son automáticamente `NEU`.** Una valoración mixta, un texto ambiguo o un texto sin información suficiente no equivalen a "sin valoración". Si no podés asignar una clase de forma defendible, **dejá `etiqueta_humana` vacía** y explicá el motivo en `nota_ambiguedad`. Ese motivo es un resultado, no un descarte.

**A quién se refiere la etiqueta.** Si la fila tiene `objeto_valorado`, la etiqueta se asigna a la valoración de **ese objeto**, no al texto entero. Si no lo tiene, se asigna al texto. Un titular puede ser favorable para un objeto y desfavorable para otro.

### Qué se anota

| Columna | Qué escribís |
|---|---|
| `etiqueta_humana` | `POS`, `NEG` o `NEU`. Vacía si no hay una clase defendible. |
| `objeto_valorado` | Viene precargado si el corpus lo trae. Corregilo si el objeto valorado es otro. |
| `evidencia_textual` | La expresión del texto que sostiene tu decisión, copiada literalmente. |
| `nota_ambiguedad` | Mixto, ambiguo, insuficiente, cita ajena, ironía, o lo que dificulte la decisión. |
| `presencia_negacion` | `SI` o `NO`, según el criterio de la sección 4. Se puede completar después. |

**No completes las etiquetas con un modelo.** La referencia humana tiene que salir de la lectura del grupo.

In [ ]:
# Columnas que se muestran a quien anota. No hay ninguna predicción.
plantilla = pd.DataFrame({"id": corpus["id"], "texto": corpus["texto"]})

if "contexto" in corpus.columns:
    plantilla["contexto"] = corpus["contexto"]

plantilla["etiqueta_humana"] = ""   # valor inicial vacío: lo completa el grupo

if "objeto_valorado" in corpus.columns:
    plantilla["objeto_valorado"] = corpus["objeto_valorado"]
else:
    plantilla["objeto_valorado"] = ""

plantilla["evidencia_textual"] = ""
plantilla["nota_ambiguedad"] = ""
plantilla["presencia_negacion"] = ""

display(plantilla.head(10))
print(f"La plantilla completa tiene {len(plantilla)} filas.")

**Leé la tabla.** Es lo que vas a completar. Mirá que no hay etiquetas ni predicciones.

La celda siguiente guarda la plantilla como CSV para completarla en una planilla (Excel, Calc, Sheets) o en un editor de texto.

**Advertencia.** El archivo contiene los textos del corpus. Si trabajás con textos sensibles, no lo compartas fuera del grupo. La celda **no sobrescribe** una plantilla que ya exista, para no borrar anotaciones en curso.

In [ ]:
CARPETA_SALIDAS.mkdir(exist_ok=True)
ruta_de_la_plantilla = CARPETA_SALIDAS / "anotaciones_plantilla.csv"

if ruta_de_la_plantilla.exists():
    print(f"Ya existe {ruta_de_la_plantilla}: no se sobrescribe.")
else:
    # utf-8-sig agrega una marca al inicio para que Excel muestre bien las tildes.
    plantilla.to_csv(ruta_de_la_plantilla, index=False, encoding="utf-8-sig")
    print(f"Plantilla guardada en {ruta_de_la_plantilla}")

### Actividad en parejas

1. **Anotar solo.** Cada persona completa su copia de la plantilla, sin hablar con su pareja y sin mirar ninguna salida automática.
2. **Comparar algunos casos.** Elijan entre 8 y 10 filas, incluyan las que cada uno marcó como difíciles, y comparen etiqueta y evidencia.
3. **Documentar los desacuerdos.** Para cada desacuerdo, escriban cuál fue la causa: ¿distinto objeto valorado, distinta lectura de una cita, ironía, texto insuficiente, protocolo poco claro?
4. **Revisar el protocolo.** Si el desacuerdo se debe a una regla ambigua, reescríbanla arriba y anoten el cambio.
5. **Construir la referencia acordada.** Vuelvan a decidir las filas en desacuerdo con el protocolo revisado. Si no logran acordar una clase, dejen la etiqueta vacía y expliquen el motivo.
6. **Guardar.** Guarden el resultado como `anotaciones/anotaciones_acordadas.csv` (la ruta está en la configuración).

**Registro de desacuerdos**

| Fila (`id`) | Etiqueta A | Etiqueta B | Causa del desacuerdo | Cómo se resolvió |
|---|---|---|---|---|
| | | | | |

**Cambios al protocolo**

| Regla original | Regla revisada | Por qué |
|---|---|---|
| | | |

### Importar las anotaciones

La importación une las anotaciones con el corpus **por `id`**, no por posición: podés reordenar o filtrar filas en la planilla sin que se mezclen. Además:

- Se detiene si hay `id` repetidos o que no existen en el corpus.
- Se detiene si hay una etiqueta distinta de `POS`, `NEG`, `NEU` o vacía.
- **No copia el texto** de la plantilla: el texto siempre sale del corpus.
- Las filas del corpus que no aparecen en el archivo quedan sin etiqueta, y se informa cuántas son.

In [ ]:
COLUMNAS_DE_ANOTACION = ["etiqueta_humana", "objeto_valorado", "evidencia_textual", "nota_ambiguedad", "presencia_negacion"]
VALORES_DE_NEGACION = ["SI", "NO", ""]


def importar_anotaciones(ruta, corpus):
    """Une las anotaciones del grupo con el corpus, usando el id.

    Recibe la ruta del CSV de anotaciones y la tabla del corpus. Devuelve el corpus
    con las columnas de anotación agregadas. Si hay ids repetidos o desconocidos, o
    etiquetas inválidas, se detiene con un mensaje.
    """
    anotaciones = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")

    for columna in ["id", "etiqueta_humana"]:
        if columna not in anotaciones.columns:
            raise ValueError(f"Al archivo de anotaciones le falta la columna '{columna}'.")

    anotaciones["id"] = anotaciones["id"].str.strip()
    if anotaciones["id"].duplicated().any():
        repetidos = anotaciones.loc[anotaciones["id"].duplicated(), "id"].tolist()
        raise ValueError(f"Hay ids repetidos en las anotaciones: {repetidos}")

    desconocidos = anotaciones.loc[~anotaciones["id"].isin(corpus["id"]), "id"].tolist()
    if desconocidos:
        raise ValueError(f"Hay ids que no existen en el corpus: {desconocidos}")

    # Normalizamos mayúsculas y espacios; después validamos.
    anotaciones["etiqueta_humana"] = anotaciones["etiqueta_humana"].str.strip().str.upper()
    inválidas = anotaciones.loc[~anotaciones["etiqueta_humana"].isin(ETIQUETAS + [""]), "etiqueta_humana"]
    if len(inválidas) > 0:
        raise ValueError(f"Etiquetas no válidas: {sorted(set(inválidas))}. Se admite POS, NEG, NEU o vacío.")

    if "presencia_negacion" in anotaciones.columns:
        negación = anotaciones["presencia_negacion"].str.strip().str.upper()
        negación = negación.replace({"SÍ": "SI"})
        if not negación.isin(VALORES_DE_NEGACION).all():
            raise ValueError("presencia_negacion solo admite SI, NO o vacío.")
        anotaciones["presencia_negacion"] = negación

    # Solo tomamos las columnas de anotación. El texto de la plantilla NO se usa.
    columnas_a_unir = ["id"]
    for columna in COLUMNAS_DE_ANOTACION:
        if columna in anotaciones.columns:
            columnas_a_unir.append(columna)
    anotaciones = anotaciones[columnas_a_unir]

    # Si el corpus ya tiene objeto_valorado, el de la anotación se guarda con otro nombre.
    anotaciones = anotaciones.rename(columns={"objeto_valorado": "objeto_valorado_humano"})

    corpus_anotado = corpus.merge(anotaciones, on="id", how="left", validate="one_to_one")
    return corpus_anotado

**Aplicación.** Usamos la función con el archivo del grupo, si existe. Si no existe, el cuaderno sigue sin referencia.

In [ ]:
COLUMNAS_NUEVAS = ["etiqueta_humana", "objeto_valorado_humano", "evidencia_textual", "nota_ambiguedad", "presencia_negacion"]

hay_archivo_de_anotaciones = RUTA_ANOTACIONES.exists()

if hay_archivo_de_anotaciones:
    corpus_anotado = importar_anotaciones(ruta=RUTA_ANOTACIONES, corpus=corpus)
    ids_anotados = pd.read_csv(RUTA_ANOTACIONES, dtype=str, keep_default_na=False, encoding="utf-8-sig")["id"].str.strip()
    cantidad_sin_fila = int((~corpus["id"].isin(ids_anotados)).sum())
    print(f"Anotaciones importadas de {RUTA_ANOTACIONES}")
    print(f"Filas del corpus que no aparecen en el archivo de anotaciones: {cantidad_sin_fila}")
else:
    corpus_anotado = corpus.copy()
    ids_anotados = pd.Series([], dtype=str)
    print(f"Todavía no existe {RUTA_ANOTACIONES}.")
    print("El cuaderno sigue en modo SIN REFERENCIA: no se calculan métricas, pero sí se comparan y auditan los métodos.")

# Las columnas que falten (por ejemplo, sin archivo) se completan con vacío.
for columna in COLUMNAS_NUEVAS:
    if columna not in corpus_anotado.columns:
        corpus_anotado[columna] = ""
    corpus_anotado[columna] = corpus_anotado[columna].fillna("")

# Motivo por el que una fila no se evalúa, cuando todavía no tiene motivo.
sin_etiqueta = corpus_anotado["etiqueta_humana"] == ""
sin_motivo = corpus_anotado["motivo_exclusion"] == ""
fila_ausente = ~corpus_anotado["id"].isin(ids_anotados)
con_nota = corpus_anotado["nota_ambiguedad"] != ""

if not hay_archivo_de_anotaciones:
    corpus_anotado.loc[sin_etiqueta & sin_motivo, "motivo_exclusion"] = "sin anotaciones cargadas"
else:
    corpus_anotado.loc[sin_etiqueta & sin_motivo & fila_ausente, "motivo_exclusion"] = "sin fila en el archivo de anotaciones"
    corpus_anotado.loc[sin_etiqueta & sin_motivo & ~fila_ausente & con_nota, "motivo_exclusion"] = "sin etiqueta: ver nota_ambiguedad"
    corpus_anotado.loc[sin_etiqueta & sin_motivo & ~fila_ausente & ~con_nota, "motivo_exclusion"] = "sin etiqueta y sin nota"

# Confirmamos que los textos no cambiaron respecto del corpus original.
assert (corpus_anotado["texto"].values == corpus["texto"].values).all(), "Los textos del corpus no pueden cambiar al importar anotaciones."
display(corpus_anotado["motivo_exclusion"].value_counts().rename("filas").to_frame())

**Leé la tabla.** Cada fila de la tabla es un motivo por el que algunas filas no se evalúan (o una celda vacía si no hay problema). Si todavía no anotaron, todas aparecen como "sin anotaciones cargadas": es lo esperado y el cuaderno sigue en modo sin referencia.

Podés ejecutar hasta acá, anotar, y recién después continuar.

## 4. Pregunta e hipótesis

**Pregunta de investigación.** Es una pregunta abierta sobre lo que queremos averiguar. Por ejemplo: "¿En qué tipo de fragmentos se separan más los tres métodos?".

**Hipótesis.** Es una afirmación que los datos pueden **contradecir**. Por ejemplo, en este corpus podría ser:

> "Los fragmentos con negación tendrán más desacuerdos entre métodos que los fragmentos sin negación."

Esa hipótesis habla de desacuerdos **entre métodos**. Aunque se cumpla, **no demuestra** por sí sola que un método interprete correctamente la negación (podrían estar equivocándose los tres, o por razones distintas).

### Qué cuenta como negación

Para contrastar esa hipótesis necesitamos una marca humana `presencia_negacion`, definida con un criterio explícito. Contar la presencia de la palabra "no" **no** es lo mismo que detectar una negación: "no" puede ser parte de un nombre, de una cita o de una pregunta retórica.

**Criterio sugerido (editable).** Marcá `SI` cuando el fragmento contiene una negación que **afecta a la expresión con la que se valora el objeto**. Marcá `NO` cuando no hay negación o cuando la negación no afecta esa valoración. Ejemplos inventados para explicar el criterio (no pertenecen al corpus):

| Fragmento inventado | Marca | Por qué |
|---|---|---|
| "La medida no resolvió el problema." | `SI` | La negación invierte la valoración de la medida. |
| "La medida resolvió el problema." | `NO` | No hay negación. |
| "Dijo que no hablaría del tema." | a discutir | La negación afecta a lo que el sujeto dice, no a la valoración. Decidan y escriban el criterio. |

**Nuestro criterio, tal como lo vamos a aplicar:**

### Plantilla de pregunta e hipótesis (editable)

| Elemento | Nuestra versión |
|---|---|
| Pregunta | |
| Hipótesis | |
| Justificación | |
| Qué observaremos | |
| Qué resultado la debilitaría | |
| Explicaciones alternativas | |
| Alcance | |

## 5. Recuperar los tres métodos

Cada método se reproduce aquí con las piezas mínimas de su cuaderno. Todos reciben **el texto original** como entrada base, y lo que cada uno hace con él (minúsculas, tokens, vocabulario) se declara en su sección.

### Método A: léxico y reglas (cuaderno 01)

**Procedencia.** Copiado del cuaderno 01 sin cambios de lógica: mismo léxico de 27 palabras, misma regla de negación (ventana de 3 tokens hacia atrás, cortada por signos y conectores) y mismo criterio para convertir puntuación en etiqueta (mayor que 0, `POS`; menor que 0, `NEG`; igual a 0, `NEU`).

**Procesamiento propio.** Pasa el texto a minúsculas y lo corta en palabras y signos.

**Límite que ya conocemos.** El léxico se armó con vocabulario de trámites y servicios. No contiene palabras propias de titulares políticos.

In [ ]:
# --- Método A: copiado del cuaderno 01 ---
import re

léxico = {
    "bueno": 1, "buena": 1, "bien": 1, "excelente": 2,
    "sencillo": 1, "sencilla": 1, "útil": 1,
    "claro": 1, "clara": 1, "rápido": 1, "rápida": 1,
    "amable": 1, "conforme": 1,
    "malo": -1, "mala": -1, "mal": -1, "pésimo": -2, "pésima": -2,
    "complicado": -1, "complicada": -1, "inútil": -1,
    "confuso": -1, "confusa": -1, "lento": -1, "lenta": -1,
    "desagradable": -1, "disconforme": -1
}

NEGADORES = {"no", "nunca", "jamás"}
LÍMITES = {".", ",", ";", ":", "!", "?", "¿", "¡", "pero", "aunque", "sin", "embargo"}
VENTANA = 3  # cuántos tokens hacia atrás mira la regla


def tokenizar(texto):
    """Corta un texto en tokens: palabras y signos, todo en minúsculas."""
    texto_en_minúsculas = texto.lower()
    tokens = re.findall(pattern=r"\w+|[^\w\s]", string=texto_en_minúsculas)
    return tokens


def hay_negación_antes(tokens, posición):
    """Dice si la palabra en esa posición queda negada por un negador cercano."""
    inicio = max(0, posición - VENTANA)
    anteriores = tokens[inicio:posición]
    for token_anterior in reversed(anteriores):
        if token_anterior in LÍMITES:
            return False
        if token_anterior in NEGADORES:
            return True
    return False

**Segunda parte del método A.** La función que calcula el valor final de cada token y la que lo convierte en etiqueta.

In [ ]:
def contribuciones_con_negación(texto):
    """Calcula el valor de cada token aplicando la regla de negación."""
    tokens = tokenizar(texto)
    filas = []
    for posición, token in enumerate(tokens):
        valor_base = léxico.get(token, 0)

        negada = False
        if valor_base != 0:
            negada = hay_negación_antes(tokens=tokens, posición=posición)

        valor_final = valor_base
        if negada:
            valor_final = -valor_base

        filas.append({"token": token, "valor_base": valor_base,
                      "negación_detectada": negada, "valor_final": valor_final})
    return pd.DataFrame(filas)


def etiqueta_desde_puntuación(puntuación):
    """Convierte una puntuación en etiqueta: POS si es mayor que 0, NEG si es menor, NEU si es 0."""
    if puntuación > 0:
        return "POS"
    if puntuación < 0:
        return "NEG"
    return "NEU"


def puntuación_con_negación(texto):
    """Suma los valores finales de un texto, con la regla de negación aplicada."""
    tabla = contribuciones_con_negación(texto)
    if len(tabla) == 0:
        return 0
    return int(tabla["valor_final"].sum())

**Una verificación de la copia.** Antes de usarla con el corpus nuevo, comprobamos que la copia se comporta como en el cuaderno 01 con oraciones de ese cuaderno.

In [ ]:
# Resultados que el cuaderno 01 obtuvo con estas oraciones (tabla de la sección 6 de ese cuaderno).
assert puntuación_con_negación("La atención no fue buena.") == -1, "La copia del método A no coincide con el cuaderno 01."
assert puntuación_con_negación("La atención no fue mala.") == 1, "La copia del método A no coincide con el cuaderno 01."
assert puntuación_con_negación("La atención fue buena.") == 1, "La copia del método A no coincide con el cuaderno 01."
print("La copia del método A coincide con el cuaderno 01 en tres casos de control.")

### Método B: TF-IDF y regresión logística (cuaderno 02)

**Procedencia.** Mismas piezas del cuaderno 02: `TfidfVectorizer(lowercase=True, ngram_range=(1, 1))` y `LogisticRegression(max_iter=1000)`.

**Entrenamiento.** Se ajusta **solo** con los 24 textos de la partición `entrenamiento` de `datos/corpus_sentimiento.csv`, que son oraciones construidas sobre trámites y servicios. **El corpus auténtico de este cuaderno no interviene en el ajuste**, y no se reentrena con la referencia humana para después evaluar sobre esas mismas filas: eso mediría la memoria del modelo, no su capacidad de generalizar.

**Procesamiento propio.** Minúsculas y vocabulario de las 60 palabras vistas en el entrenamiento. Las palabras que no están en ese vocabulario **no existen** para el modelo.

**Para auditar.** Además de la etiqueta, contamos cuántas palabras de cada texto conoce el modelo.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

corpus_de_entrenamiento = pd.read_csv(RUTA_ENTRENAMIENTO)
entrenamiento = corpus_de_entrenamiento.loc[corpus_de_entrenamiento["partición"] == "entrenamiento"]
textos_de_entrenamiento = entrenamiento["texto"]
etiquetas_de_entrenamiento = entrenamiento["etiqueta_referencia"]

# El corpus auténtico no puede estar dentro del entrenamiento.
textos_en_común = set(textos_de_entrenamiento) & set(corpus["texto"])
assert len(textos_en_común) == 0, f"Hay textos del corpus nuevo dentro del entrenamiento: {textos_en_común}"

vectorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 1))
vectorizador.fit(textos_de_entrenamiento)
tfidf_de_entrenamiento = vectorizador.transform(textos_de_entrenamiento)

clasificador_tfidf = LogisticRegression(max_iter=1000)
clasificador_tfidf.fit(X=tfidf_de_entrenamiento, y=etiquetas_de_entrenamiento)

vocabulario = set(vectorizador.get_feature_names_out())
print(f"Método B entrenado con {len(entrenamiento)} textos construidos y {len(vocabulario)} palabras de vocabulario.")

### Método C: modelo preentrenado (cuaderno 03)

**Procedencia.** El mismo `create_analyzer(task="sentiment", lang="es")` del cuaderno 03.

**Procesamiento propio.** Lo hace el propio modelo, que decide cómo segmentar el texto. No le aplicamos limpieza previa.

**Qué devuelve.** Una etiqueta y cómo reparte el modelo su 100 % entre `NEG`, `NEU` y `POS`. La probabilidad máxima **no es certeza** ni indica que la etiqueta sea correcta.

**Qué registramos.** Versiones de las librerías y el identificador del modelo cargado.

**Descarga.** La primera vez que se usa, el modelo se descarga de Internet (ver `ANTES_DE_LA_CLASE.md`). Después funciona sin conexión.

In [ ]:
try:
    from pysentimiento import create_analyzer
except ImportError:
    print("Falta pysentimiento. Instalalo con: uv pip install -r requirements.txt")
    raise

analizador = create_analyzer(task="sentiment", lang="es")
configuración_del_modelo = analizador.model.config
identificador_del_modelo = str(getattr(configuración_del_modelo, "_name_or_path", "No informado"))

assert set(configuración_del_modelo.id2label.values()) == set(ETIQUETAS), "El modelo debería responder NEG, NEU o POS."

versiones = {"python": sys.version.split()[0]}
for paquete in ["pandas", "scikit-learn", "torch", "transformers", "pysentimiento"]:
    versiones[paquete] = importlib.metadata.version(paquete)
print("Modelo cargado:", identificador_del_modelo)
print("Versiones:", versiones)

**Leé el resultado.** Anotá el identificador del modelo y las versiones: sin ellas, nadie puede repetir tu análisis.

## 6. Comparación

**Operación.** Aplicamos los tres métodos **a los mismos registros**: todas las filas con texto. Después armamos una tabla única.

Distinguimos dos cosas que se confunden fácilmente:

- **Acuerdo entre métodos:** los métodos dan la misma etiqueta entre sí.
- **Coincidencia con la referencia humana:** un método da la misma etiqueta que el grupo.

El acuerdo entre métodos **no demuestra corrección**: tres métodos pueden coincidir y estar equivocados.

In [ ]:
# Solo se clasifican las filas con texto. Las demás quedan en la tabla, marcadas.
a_clasificar = corpus_anotado.loc[corpus_anotado["texto"].str.strip() != ""]

# Método A: puntuación y etiqueta.
puntuaciones_lexico = a_clasificar["texto"].apply(puntuación_con_negación)
etiquetas_lexico = puntuaciones_lexico.apply(etiqueta_desde_puntuación)

# Método B: cada texto pasa por el vocabulario del entrenamiento (transform) y se clasifica.
tfidf_del_corpus = vectorizador.transform(a_clasificar["texto"])
etiquetas_tfidf = pd.Series(clasificador_tfidf.predict(tfidf_del_corpus), index=a_clasificar.index)

print(f"Textos clasificados por los métodos A y B: {len(a_clasificar)}")

**Método C y diagnóstico del método B.** Aplicamos el modelo preentrenado y contamos cuántas palabras de cada texto conoce el método B.

In [ ]:
def contar_palabras_conocidas(texto):
    """Cuenta cuántas palabras del texto están en el vocabulario del método B."""
    cortar_en_palabras = vectorizador.build_analyzer()
    palabras = cortar_en_palabras(texto)
    conocidas = 0
    for palabra in palabras:
        if palabra in vocabulario:
            conocidas = conocidas + 1
    return conocidas


def analizar_con_pysentimiento(textos):
    """Aplica el modelo a cada texto. Devuelve etiqueta y probabilidades, con el mismo índice."""
    filas = []
    for texto in textos:
        resultado = analizador.predict(texto)
        filas.append({
            "pred_pysentimiento": resultado.output,
            "prob_NEG": resultado.probas["NEG"],
            "prob_NEU": resultado.probas["NEU"],
            "prob_POS": resultado.probas["POS"],
        })
    return pd.DataFrame(filas, index=textos.index)


resultados_pysentimiento = analizar_con_pysentimiento(a_clasificar["texto"])
palabras_conocidas_tfidf = a_clasificar["texto"].apply(contar_palabras_conocidas)
print(f"Textos clasificados por el método C: {len(resultados_pysentimiento)}")

**Tabla única.** Juntamos las predicciones de los tres métodos con la anotación y los datos del corpus.

In [ ]:
comparacion = corpus_anotado.copy()
comparacion["pred_lexico"] = etiquetas_lexico
comparacion["pred_tfidf"] = etiquetas_tfidf
comparacion["pred_pysentimiento"] = resultados_pysentimiento["pred_pysentimiento"]
comparacion["puntuacion_lexico"] = puntuaciones_lexico
comparacion["palabras_conocidas_tfidf"] = palabras_conocidas_tfidf
for columna in ["prob_NEG", "prob_NEU", "prob_POS"]:
    comparacion[columna] = resultados_pysentimiento[columna]

# Probabilidad máxima del modelo: cuánto concentra en una clase. No es certeza.
comparacion["prob_maxima_pysentimiento"] = resultados_pysentimiento[["prob_NEG", "prob_NEU", "prob_POS"]].max(axis=1)

# Las filas sin texto no tienen predicciones: sus columnas de texto quedan vacías.
for columna in ["pred_lexico", "pred_tfidf", "pred_pysentimiento"]:
    comparacion[columna] = comparacion[columna].fillna("")

columnas_a_mostrar = [
    "id", "texto", "etiqueta_humana", "pred_lexico", "pred_tfidf", "pred_pysentimiento",
    "puntuacion_lexico", "prob_NEG", "prob_NEU", "prob_POS", "nota_ambiguedad", "id_documento",
]
display(comparacion[columnas_a_mostrar].round(2))

**Leé la tabla.** `etiqueta_humana` vacía significa que todavía no hay referencia para esa fila (o que el grupo decidió no asignarle una clase). `puntuacion_lexico` es el número que da el método A antes de convertirlo en etiqueta. Las tres columnas `prob_` son cómo reparte el modelo C su 100 %.

Fijate también en `palabras_conocidas_tfidf` (en la tabla completa `comparacion`): si vale 0, el método B no vio ninguna palabra de su vocabulario y decide solo con lo que tiene por defecto.

In [ ]:
# Cuántas etiquetas distintas dan los tres métodos para un mismo texto.
def contar_etiquetas_distintas(fila):
    """Devuelve 1 si los tres métodos coinciden, 2 si coinciden dos, 3 si los tres difieren."""
    etiquetas = {fila["pred_lexico"], fila["pred_tfidf"], fila["pred_pysentimiento"]}
    return len(etiquetas)


descripciones = {1: "los tres coinciden", 2: "coinciden dos", 3: "los tres difieren"}

# apply con axis=1 aplica la función fila por fila.
distintas = comparacion.loc[a_clasificar.index].apply(contar_etiquetas_distintas, axis=1)
comparacion["acuerdo_entre_metodos"] = distintas.map(descripciones)
comparacion["acuerdo_entre_metodos"] = comparacion["acuerdo_entre_metodos"].fillna("")

cantidades = comparacion["acuerdo_entre_metodos"].value_counts()
filas_del_acuerdo = []
for situación in descripciones.values():
    filas_del_acuerdo.append({"situación": situación, "fragmentos": int(cantidades.get(situación, 0))})
tabla_de_acuerdo = pd.DataFrame(filas_del_acuerdo)
dibujar_barras(
    tabla=tabla_de_acuerdo, columna_nombre="situación", columna_valor="fragmentos",
    título="Acuerdo entre los tres métodos (no es lo mismo que acierto)", color=COLOR_GENERAL,
)

**Leé el gráfico.** Mide cuántos fragmentos reciben la misma etiqueta de los tres métodos, de dos o de ninguno. **No mide aciertos.** Anotá:

- **Observación:** ¿en cuántos fragmentos coinciden los tres?
- **Interpretación posible:** ¿qué podría explicar los desacuerdos?
- **Límite:** ¿qué no podemos decir mirando solo este gráfico?

### Distribución de etiquetas por método

Antes de evaluar, mirá qué etiquetas usa cada método. Un método que casi nunca dice `NEU`, por ejemplo, tiene una distribución muy distinta de la del grupo.

In [ ]:
distribución = pd.DataFrame({
    "método A: léxico": comparacion["pred_lexico"].value_counts(),
    "método B: TF-IDF": comparacion["pred_tfidf"].value_counts(),
    "método C: pysentimiento": comparacion["pred_pysentimiento"].value_counts(),
    "referencia humana": comparacion["etiqueta_humana"].value_counts(),
})
distribución = distribución.drop(index="", errors="ignore")
distribución = distribución.reindex(ETIQUETAS).fillna(0).astype(int)
display(distribución)

**Leé la tabla.** Cada columna cuenta cuántas veces dice una etiqueta cada método. La columna de la referencia humana solo cuenta las filas que el grupo etiquetó.

**Guardar la comparación.** La celda siguiente guarda la tabla en `salidas/` junto con el registro del entorno.

**Advertencia.** El archivo incluye los textos del corpus. No lo compartas fuera del grupo si trabajás con textos sensibles.

In [ ]:
import json

CARPETA_SALIDAS.mkdir(exist_ok=True)
comparacion.to_csv(CARPETA_SALIDAS / "comparacion_metodos.csv", index=False, encoding="utf-8")

registro = {
    "corpus": str(RUTA_CORPUS),
    "fragmentos": len(comparacion),
    "modelo_pysentimiento": identificador_del_modelo,
    "versiones": versiones,
    "entrenamiento_metodo_B": str(RUTA_ENTRENAMIENTO),
}
with open(CARPETA_SALIDAS / "registro_cuaderno_04.json", "w", encoding="utf-8") as archivo:
    json.dump(registro, archivo, ensure_ascii=False, indent=2)
print("Guardado en", CARPETA_SALIDAS)

## 7. Evaluación

**Pregunta.** Contra la referencia que construyó el grupo, ¿qué clasificaciones coinciden y cuáles no?

**Condiciones.**

- Se evalúan **solo** las filas con referencia humana válida (`POS`, `NEG` o `NEU`).
- Los tres métodos se evalúan sobre **las mismas filas**.
- Con menos de `MINIMO_EVALUADOS` filas (ver configuración), no calculamos métricas: serían una ilusión de precisión.
- No se hacen pruebas de significación ni afirmaciones poblacionales.

In [ ]:
tiene_referencia = comparacion["etiqueta_humana"].isin(ETIQUETAS)
tiene_predicciones = comparacion["pred_lexico"] != ""
evaluados = comparacion.loc[tiene_referencia & tiene_predicciones]

cantidad_total = len(comparacion)
cantidad_evaluada = len(evaluados)
hay_referencia = cantidad_evaluada >= MINIMO_EVALUADOS

print(f"Registros en total: {cantidad_total}")
print(f"Registros evaluados (con referencia humana válida): {cantidad_evaluada}")
print(f"Registros sin referencia: {cantidad_total - cantidad_evaluada}")
print()
print("Motivos por los que un registro no se evalúa:")
display(comparacion.loc[~(tiene_referencia & tiene_predicciones), "motivo_exclusion"].value_counts().rename("registros").to_frame())

if hay_referencia:
    print("Distribución de clases en la referencia humana:")
    display(evaluados["etiqueta_humana"].value_counts().reindex(ETIQUETAS).fillna(0).astype(int).rename("ejemplos").to_frame())
else:
    print(f"MODO SIN REFERENCIA: se necesitan al menos {MINIMO_EVALUADOS} registros anotados para calcular métricas.")
    print("No se muestran métricas. Las secciones de auditoría siguen disponibles.")

**Leé el resultado.** Fijate si alguna clase tiene pocos ejemplos o ninguno. **Una clase sin ejemplos de referencia no tiene métricas interpretables**: no es evidencia de buen ni de mal rendimiento.

In [ ]:
from sklearn.metrics import classification_report

MÉTODOS = {
    "lexico": "Método A: léxico y reglas",
    "tfidf": "Método B: TF-IDF y regresión logística",
    "pysentimiento": "Método C: pysentimiento",
}


def resumir_métricas(referencia, predicción):
    """Calcula precisión, exhaustividad y F1 por clase, y el macro-F1.

    Las clases sin ejemplos de referencia quedan sin métricas: no son interpretables.
    El macro-F1 promedia solo las clases que sí tienen ejemplos de referencia.
    """
    informe = classification_report(
        y_true=referencia, y_pred=predicción, labels=ETIQUETAS,
        output_dict=True, zero_division=0,
    )
    filas = []
    suma_f1 = 0.0
    clases_con_ejemplos = 0
    for clase in ETIQUETAS:
        cantidad = int(informe[clase]["support"])
        fila = {"clase": clase, "ejemplos de referencia": cantidad,
                "precisión": None, "exhaustividad": None, "F1": None}
        if cantidad > 0:
            fila["precisión"] = informe[clase]["precision"]
            fila["exhaustividad"] = informe[clase]["recall"]
            fila["F1"] = informe[clase]["f1-score"]
            suma_f1 = suma_f1 + informe[clase]["f1-score"]
            clases_con_ejemplos = clases_con_ejemplos + 1
        filas.append(fila)
    macro_f1 = None
    if clases_con_ejemplos > 0:
        macro_f1 = suma_f1 / clases_con_ejemplos
    return pd.DataFrame(filas), macro_f1, clases_con_ejemplos

**Aplicación.** Para cada método calculamos la matriz de confusión y las métricas sobre las mismas filas.

In [ ]:
resumen_final = []

if hay_referencia:
    for clave, nombre in MÉTODOS.items():
        referencia = evaluados["etiqueta_humana"]
        predicción = evaluados[f"pred_{clave}"]

        dibujar_matriz_de_confusión(referencia=referencia, predicción=predicción, título=nombre)

        tabla_por_clase, macro_f1, clases_con_ejemplos = resumir_métricas(referencia=referencia, predicción=predicción)
        display(tabla_por_clase.round(2))
        if clases_con_ejemplos < len(ETIQUETAS):
            print(f"Advertencia: la referencia solo tiene ejemplos de {clases_con_ejemplos} de {len(ETIQUETAS)} clases. "
                  "El macro-F1 promedia solo esas clases.")
        coinciden = int((referencia == predicción).sum())
        resumen_final.append({"método": nombre, "evaluados": len(evaluados),
                              "coinciden con la referencia": coinciden, "macro-F1": macro_f1})
else:
    print("Sin métricas: no hay suficientes registros con referencia humana.")

**Leé las matrices y las tablas.**

- En la matriz, las filas son la referencia humana y las columnas, la respuesta del método. La diagonal son las coincidencias.
- **Precisión:** de lo que el método puso en una clase, cuánto coincide con la referencia. **Exhaustividad:** de lo que la referencia puso en una clase, cuánto encontró el método. **F1** combina ambas.
- Una celda vacía en una clase significa que la referencia no tiene ejemplos de ella.

**Observación, interpretación y límite.** Escribí tres frases distintas: qué muestran los números, qué podría explicarlo y qué no permiten afirmar. Con unas pocas decenas de registros, un solo caso cambia mucho cada porcentaje.

In [ ]:
if hay_referencia:
    tabla_resumen = pd.DataFrame(resumen_final)
    display(tabla_resumen.round(2))
else:
    print("Sin resumen: no hay suficientes registros con referencia humana.")

**Nota para interpretar.** Esta tabla compara métodos **sobre estos registros y esta anotación**. No dice cuál método es mejor en general, ni cuántos aciertos tendría con otros textos.

Respuestas del grupo:

| Pregunta | Nuestra respuesta |
|---|---|
| ¿Qué método coincide más con la referencia? | |
| ¿Esa diferencia es grande o cambiaría con uno o dos casos distintos? | |
| ¿Qué clase está peor representada en la referencia? | |

## 8. Auditoría de casos

Los números resumen. Para entender qué pasó hay que **leer los textos completos**. Las tablas siguientes muestran el texto sin recortes.

Hay cuatro tablas. Las que dependen de la referencia humana solo aparecen si hay anotaciones.

In [ ]:
def mostrar_completo(tabla):
    """Muestra una tabla con los textos completos, sin recortar."""
    with pd.option_context("display.max_colwidth", None):
        display(tabla)


# Columnas que se muestran en la auditoría. Las opcionales aparecen solo si el corpus las tiene.
COLUMNAS_DE_AUDITORÍA = []
for columna in ["id", "texto", "objeto_valorado", "objeto_valorado_humano", "etiqueta_humana",
                "pred_lexico", "pred_tfidf", "pred_pysentimiento", "nota_ambiguedad"]:
    if columna in comparacion.columns:
        COLUMNAS_DE_AUDITORÍA.append(columna)

desacuerdan = comparacion["acuerdo_entre_metodos"].isin(["coinciden dos", "los tres difieren"])
tabla_1 = comparacion.loc[desacuerdan, COLUMNAS_DE_AUDITORÍA]
print(f"Tabla 1: fragmentos en los que los métodos discrepan ({len(tabla_1)})")
mostrar_completo(tabla_1)

**Tablas 2, 3 y 4.** Consenso contra la referencia, casos ambiguos y desacuerdos de pysentimiento con probabilidad alta.

In [ ]:
if hay_referencia:
    coinciden_los_tres = comparacion["acuerdo_entre_metodos"] == "los tres coinciden"
    difiere_de_la_referencia = comparacion["pred_lexico"] != comparacion["etiqueta_humana"]
    tabla_2 = comparacion.loc[tiene_referencia & coinciden_los_tres & difiere_de_la_referencia, COLUMNAS_DE_AUDITORÍA]
    print(f"Tabla 2: los tres métodos coinciden entre sí y difieren de la referencia ({len(tabla_2)})")
    mostrar_completo(tabla_2)

    tabla_3 = comparacion.loc[comparacion["nota_ambiguedad"] != "", COLUMNAS_DE_AUDITORÍA]
    print(f"Tabla 3: fragmentos con nota de ambigüedad ({len(tabla_3)})")
    mostrar_completo(tabla_3)

    seguros = comparacion["prob_maxima_pysentimiento"] >= UMBRAL_PROBABILIDAD_ALTA
    difiere_pysentimiento = comparacion["pred_pysentimiento"] != comparacion["etiqueta_humana"]
    tabla_4 = comparacion.loc[tiene_referencia & seguros & difiere_pysentimiento, COLUMNAS_DE_AUDITORÍA + ["prob_maxima_pysentimiento"]]
    print(f"Tabla 4: desacuerdos de pysentimiento con probabilidad máxima de {UMBRAL_PROBABILIDAD_ALTA:.0%} o más ({len(tabla_4)})")
    mostrar_completo(tabla_4)
else:
    print("Las tablas 2, 3 y 4 necesitan una referencia humana: completá y guardá las anotaciones y volvé a ejecutar desde la sección 3.")

**Gráfico de la tabla 4.** Cómo reparte el modelo su 100 % en esos desacuerdos.

In [ ]:
if hay_referencia and len(tabla_4) > 0:
    para_graficar = tabla_4.copy()
    para_graficar = para_graficar.merge(comparacion[["id", "prob_NEG", "prob_NEU", "prob_POS"]], on="id")
    para_graficar["rótulo"] = para_graficar["id"] + " (ref. " + para_graficar["etiqueta_humana"] + "): " + para_graficar["texto"]
    dibujar_probabilidades(
        tabla=para_graficar, columna_texto="rótulo",
        título="Tabla 4: cómo reparte pysentimiento su 100 % en sus desacuerdos de alta probabilidad",
    )

**Leé la tabla 4 y el gráfico.** Una probabilidad alta indica que el modelo concentra su respuesta en una clase. No indica que esa clase sea la correcta. Si hay desacuerdos con probabilidad alta, son los casos más informativos para preguntarse si el problema es del método, de la anotación o de la definición de la tarea.

### ✎ Consignas de auditoría

Para cada caso que analices, respondé con evidencia textual:

1. ¿Qué objeto se valora?
2. ¿Quién sostiene la valoración? ¿Quién escribe el titular, o alguien a quien se cita?
3. ¿Hay cita, ironía, negación o varias voces?
4. ¿El desacuerdo es **del método**, **de la anotación** o **de la definición de tarea**? Explicá por qué.
5. ¿Qué expresión del texto sostiene tu interpretación? Copiala literalmente.
6. ¿Qué se pierde al asignar una única clase?

### ✎ Análisis escrito (al menos tres casos)

Elegí al menos tres casos de las tablas. Incluí uno en el que los métodos discrepan y uno en el que difieren de la referencia. **Estas respuestas las escribe el grupo; el cuaderno no las genera.**

**Caso 1**

- `id`:
- Texto completo:
- Objeto valorado y quién lo valora:
- ¿Del método, de la anotación o de la definición de tarea?:
- Expresión que sostiene la interpretación:
- Qué se pierde con una sola clase:

**Caso 2**

- `id`:
- Texto completo:
- Objeto valorado y quién lo valora:
- ¿Del método, de la anotación o de la definición de tarea?:
- Expresión que sostiene la interpretación:
- Qué se pierde con una sola clase:

**Caso 3**

- `id`:
- Texto completo:
- Objeto valorado y quién lo valora:
- ¿Del método, de la anotación o de la definición de tarea?:
- Expresión que sostiene la interpretación:
- Qué se pierde con una sola clase:

## 9. Contrastar la hipótesis

Si tu hipótesis usa `presencia_negacion`, el contraste se hace con **las marcas humanas** de esa columna, aplicando el criterio que escribieron en la sección 4. No usamos la presencia de la palabra "no" como sustituto.

**Qué comparamos.** Cuántos fragmentos tienen desacuerdo entre métodos en cada grupo, con su denominador y su proporción. Un desacuerdo es un fragmento en el que los tres métodos **no** dan la misma etiqueta.

**Qué no hacemos.** No usamos lenguaje causal ("la negación causa errores"), y no interpretamos la ausencia de desacuerdo como comprensión correcta: tres métodos pueden coincidir y estar equivocados.

In [ ]:
con_texto = comparacion.loc[tiene_predicciones]
GRUPOS = {"SI": "con negación (SI)", "NO": "sin negación (NO)", "": "sin marca"}

filas_del_contraste = []
for marca, nombre in GRUPOS.items():
    grupo = con_texto.loc[con_texto["presencia_negacion"] == marca]
    con_desacuerdo = grupo.loc[grupo["acuerdo_entre_metodos"] != "los tres coinciden"]
    proporción = None
    if len(grupo) > 0:
        proporción = len(con_desacuerdo) / len(grupo)
    filas_del_contraste.append({
        "grupo": nombre,
        "fragmentos (denominador)": len(grupo),
        "con desacuerdo entre métodos": len(con_desacuerdo),
        "proporción": proporción,
    })

tabla_del_contraste = pd.DataFrame(filas_del_contraste)
display(tabla_del_contraste.round(2))

hay_marcas = (con_texto["presencia_negacion"] != "").any()
if not hay_marcas:
    print("Todavía no hay marcas de presencia_negacion: completá esa columna en las anotaciones para contrastar la hipótesis.")

**Leé la tabla.** Mirá siempre el denominador antes que la proporción.

- **Observación:** ¿en cuál grupo hay proporcionalmente más desacuerdos? ¿Con cuántos fragmentos en cada grupo?
- **Interpretación posible:** ¿qué otras diferencias entre ambos grupos (largo del titular, objeto, tema) podrían explicarlo?
- **Límite:** con grupos chicos, un solo fragmento cambia la proporción de forma notable. Y el grupo "sin marca" no se puede ignorar: indica cuánto de la comparación está sin completar.

### Decisión sobre la hipótesis

Elegí una opción y justificala con la tabla y con al menos un fragmento de la sección 8:

- [ ] Se acepta **provisionalmente** (y por qué es solo provisional).
- [ ] Se **revisa** (cómo la reformulamos).
- [ ] Se **descarta** (qué la contradijo).

**Justificación:**

## 10. Una segunda referencia: la etiqueta original del corpus

**Esta sección se ejecuta solo después de que el grupo terminó de anotar.** Si la ejecutás antes, conocés la etiqueta original antes de decidir la tuya y la comparación pierde sentido.

El corpus trae su propia etiqueta, hecha por otras personas con un criterio que no conocemos en detalle. La comparamos **como segunda referencia, no como verdad**:

- La referencia del grupo y la del corpus pueden diferir porque el protocolo es distinto, no porque una esté mal.
- Las etiquetas originales son de **valoración dirigida a un objeto**. Los tres métodos clasifican el texto completo y **no ven** el objeto: esa diferencia de tarea es un hallazgo de la auditoría, no un defecto de quien anotó.
- Los datos no permiten saber con qué acuerdo entre anotadores se hicieron.

In [ ]:
if not hay_referencia:
    print("Sección 10 omitida: primero hay que tener anotaciones del grupo (sección 3).")
elif not RUTA_REFERENCIA_ORIGINAL.exists():
    print(f"No se encontró {RUTA_REFERENCIA_ORIGINAL}: este corpus no trae una referencia externa.")
else:
    referencia_externa = pd.read_csv(RUTA_REFERENCIA_ORIGINAL, dtype=str, keep_default_na=False)
    assert referencia_externa["id"].is_unique, "La referencia externa tiene ids repetidos."
    contra_el_corpus = evaluados.merge(referencia_externa, on="id", how="left", validate="one_to_one")
    contra_el_corpus = contra_el_corpus.dropna(subset=["etiqueta_corpus"])
    print(f"Filas con las dos referencias: {len(contra_el_corpus)} de {len(evaluados)}")

    filas = []
    for nombre, columna in [("referencia del grupo", "etiqueta_humana"), ("método A", "pred_lexico"),
                            ("método B", "pred_tfidf"), ("método C", "pred_pysentimiento")]:
        coinciden = int((contra_el_corpus[columna] == contra_el_corpus["etiqueta_corpus"]).sum())
        filas.append({"se compara con la etiqueta original": nombre, "coinciden": coinciden,
                      "sobre": len(contra_el_corpus)})
    display(pd.DataFrame(filas))

    dibujar_matriz_de_confusión(
        referencia=contra_el_corpus["etiqueta_corpus"], predicción=contra_el_corpus["etiqueta_humana"],
        título="Etiqueta original del corpus y referencia del grupo",
        nombre_filas="Etiqueta original del corpus", nombre_columnas="Referencia del grupo",
    )

**Leé la tabla y la matriz.**

- **Observación:** ¿cuánto coincide la referencia del grupo con la original? ¿Cuánto coinciden los métodos con cada una?
- **Interpretación posible:** ¿qué diferencias de tarea o de protocolo podrían explicar las discrepancias?
- **Límite:** que dos referencias coincidan no prueba que sean correctas, y que difieran no prueba que una sea incorrecta.

Elegí un caso en el que la referencia del grupo y la original difieran y explicá si la diferencia viene del objeto valorado, de la lectura del titular o del protocolo.

## 11. Ensayo computacional final

Escribí una conclusión breve que siga esta plantilla. Cada punto se apoya en una tabla, un gráfico o un fragmento de este cuaderno.

| Elemento | Qué escribir |
|---|---|
| **Pregunta** | La que formularon en la sección 4. |
| **Corpus y selección** | Qué textos, de dónde, cuántos fragmentos y cuántos documentos, y cómo se eligieron. |
| **Protocolo** | Cómo anotaron, qué cambiaron del protocolo inicial y qué hicieron con los casos no clasificables. |
| **Hipótesis** | La formulada, y si se acepta provisionalmente, se revisa o se descarta. |
| **Resultado observado** | Qué mostraron las tablas (con denominadores). Solo lo observado. |
| **Dos fragmentos como evidencia** | Dos textos completos que sostienen su interpretación, con la expresión clave. |
| **Un contraejemplo** | Un texto que no encaja con la interpretación. |
| **Explicación alternativa** | Otra explicación posible de lo observado. |
| **Límites** | Qué no permiten afirmar el corpus, la anotación y los métodos. |
| **Recomendación de uso** | Para qué tarea y bajo qué condiciones usarían cada método, si es que alguno. |

### Qué no se puede escribir

Estas frases **no están respaldadas** por lo que hicimos:

- "Esto demuestra cómo se sienten los argentinos."
- "El método X es el mejor." (sin decir mejor para qué, sobre qué textos y con qué referencia).
- "Los tres métodos coinciden, entonces la clasificación es correcta."
- "La negación hace que los métodos fallen." (lenguaje causal sin un diseño que lo sostenga).
- Cualquier afirmación sobre candidatos, medios o electores a partir de esta selección.

**Ensayo del grupo:**

## Cierre

Este cuaderno cerró la unidad con una pregunta distinta de las anteriores: no cuál método clasifica "mejor", sino qué evidencia sostiene cada clasificación y qué conclusiones permiten o no los datos.

Las limitaciones que observaron nos llevan a una pregunta nueva: cómo representar las relaciones entre palabras y entre textos más allá de las reglas y de las coincidencias de vocabulario. Explorar esa pregunta exige, igual que acá, decidir qué se mide, contra qué referencia y con qué límites; una representación nueva no elimina esa exigencia.